# Notebook 02 — Data Cleaning & ETL Pipeline
**NST DVA Capstone 2 | Healthcare Sector — Diabetic Patient Readmission**

| Field | Detail |
|---|---|
| **Input** | `data/raw/diabetic_data.csv` |
| **Output** | `data/processed/diabetic_data_clean.csv` |
| **Pipeline Stage** | Extract → **Clean & Transform** → Analyze → Load |
| **Raw Shape** | 101,766 rows × 50 columns |

> ⚠️ **Rule:** The raw file in `data/raw/` is NEVER modified. This notebook only reads it.

---

## Cell 1 — Install & Import Libraries

In [1]:
# Uncomment if running on Google Colab
# !pip install pandas numpy matplotlib seaborn --quiet

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries loaded successfully.')
print(f'  pandas  : {pd.__version__}')
print(f'  numpy   : {np.__version__}')

Libraries loaded successfully.
  pandas  : 3.0.2
  numpy   : 2.4.4


## Cell 2 — Path Configuration

In [2]:
RAW_PATH       = '../data/raw/diabetic_data.csv'
PROCESSED_DIR  = '../data/processed/'
PROCESSED_PATH = PROCESSED_DIR + 'diabetic_data_clean.csv'
LOG_PATH       = PROCESSED_DIR + '02_cleaning_log.txt'

os.makedirs(PROCESSED_DIR, exist_ok=True)

if not os.path.exists(RAW_PATH):
    raise FileNotFoundError(f'Raw file not found: {RAW_PATH}. Place diabetic_data.csv in data/raw/')

print(f'Raw file   : {RAW_PATH}  ({os.path.getsize(RAW_PATH)/1e6:.1f} MB)')
print(f'Output dir : {PROCESSED_DIR}')

Raw file   : ../data/raw/diabetic_data.csv  (19.2 MB)
Output dir : ../data/processed/


## Cell 3 — Load Raw Dataset

In [3]:
# '?' is the missing-value marker used throughout this UCI dataset
df = pd.read_csv(RAW_PATH, na_values=['?', ''], low_memory=False)

RAW_ROWS, RAW_COLS = df.shape
print(f'Loaded: {RAW_ROWS:,} rows x {RAW_COLS} columns')

# Transformation log — every step appended here, written to file at the end
log = []
def record(step, description, rows_before=None, rows_after=None):
    msg = f'[STEP {step:02d}] {description}'
    if rows_before is not None:
        removed = rows_before - rows_after
        msg += f'  |  rows: {rows_before:,} -> {rows_after:,}  (removed {removed:,})'
    log.append(msg)
    print(msg)

record(0, f'Raw dataset loaded: {RAW_ROWS:,} rows x {RAW_COLS} cols')

Loaded: 101,766 rows x 50 columns
[STEP 00] Raw dataset loaded: 101,766 rows x 50 cols


## Cell 4 — Pre-Cleaning Snapshot
Quick audit of the raw data before any changes are made.

In [4]:
null_pct = (df.isnull().mean() * 100).round(1)
audit = pd.DataFrame({
    'dtype'      : df.dtypes,
    'null_count' : df.isnull().sum(),
    'null_%'     : null_pct,
    'unique'     : df.nunique()
})

print('=== PRE-CLEANING AUDIT ===')
print(audit.to_string())
print(f'\nTotal nulls: {df.isnull().sum().sum():,}')

=== PRE-CLEANING AUDIT ===
                          dtype  null_count  null_%  unique
encounter_id              int64           0    0.00  101766
patient_nbr               int64           0    0.00   71518
race                        str        2273    2.20       5
gender                      str           0    0.00       3
age                         str           0    0.00      10
weight                      str       98569   96.90       9
admission_type_id         int64           0    0.00       8
discharge_disposition_id  int64           0    0.00      26
admission_source_id       int64           0    0.00      17
time_in_hospital          int64           0    0.00      14
payer_code                  str       40256   39.60      17
medical_specialty           str       49949   49.10      72
num_lab_procedures        int64           0    0.00     118
num_procedures            int64           0    0.00       7
num_medications           int64           0    0.00      75
number_outpat

## Cell 5 — STEP 01: Drop Zero-Variance Columns

**Why:** `examide` and `citoglipton` contain only one unique value (`No`) across all 101,766 rows.
Columns with zero variance carry no information and add unnecessary width to the dataset.

In [5]:
zero_var_cols = [c for c in df.columns if df[c].nunique(dropna=True) == 1]
print(f'Zero-variance columns found: {zero_var_cols}')

df.drop(columns=zero_var_cols, inplace=True)
record(1, f'Dropped zero-variance columns: {zero_var_cols}')
print(f'Shape after step 01: {df.shape}')

Zero-variance columns found: ['examide', 'citoglipton']
[STEP 01] Dropped zero-variance columns: ['examide', 'citoglipton']
Shape after step 01: (101766, 48)


## Cell 6 — STEP 02: Drop `weight` Column

**Why:** `weight` has 96.9% missing values (98,569 out of 101,766 rows).
At this missingness level, any imputation strategy would fabricate data rather than fill it.
The column is dropped entirely. Weight is not a KPI for this readmission study.

In [6]:
weight_miss = df['weight'].isnull().mean() * 100
print(f'weight missingness: {weight_miss:.1f}%')

df.drop(columns=['weight'], inplace=True)
record(2, f'Dropped weight column ({weight_miss:.1f}% missing — imputation not viable)')
print(f'Shape after step 02: {df.shape}')

weight missingness: 96.9%
[STEP 02] Dropped weight column (96.9% missing — imputation not viable)
Shape after step 02: (101766, 47)


## Cell 7 — STEP 03: Remove Deceased & Hospice Patients

**Why:** Readmission analysis only makes sense for patients who were discharged alive.
Discharge disposition IDs 11, 13, 14, 19, 20, 21 correspond to death or hospice transfer
in the ICD coding system. Including them would artificially lower the readmission rate.

In [7]:
EXPIRED_CODES = [11, 13, 14, 19, 20, 21]
before = len(df)

df = df[~df['discharge_disposition_id'].isin(EXPIRED_CODES)].copy()

record(3, f'Removed deceased/hospice patients (discharge IDs: {EXPIRED_CODES})', before, len(df))

[STEP 03] Removed deceased/hospice patients (discharge IDs: [11, 13, 14, 19, 20, 21])  |  rows: 101,766 -> 99,343  (removed 2,423)


## Cell 8 — STEP 04: Deduplicate — Keep First Encounter Per Patient

**Why:** 71,518 patients appear more than once (multiple hospital visits).
In a readmission study, using multiple rows for the same patient leaks information
(a later visit IS the readmission event). We keep only the earliest encounter per patient.

In [8]:
before = len(df)
multi_enc = df['patient_nbr'].duplicated().sum()
print(f'Patients with multiple encounters: {multi_enc:,}')

# Sort by encounter_id (ascending = chronological) then keep first per patient
df = df.sort_values('encounter_id').drop_duplicates(subset='patient_nbr', keep='first').copy()

record(4, 'Kept first encounter per patient_nbr to prevent data leakage in readmission analysis', before, len(df))
print(f'Unique patients remaining: {df["patient_nbr"].nunique():,}')

Patients with multiple encounters: 29,353
[STEP 04] Kept first encounter per patient_nbr to prevent data leakage in readmission analysis  |  rows: 99,343 -> 69,990  (removed 29,353)
Unique patients remaining: 69,990


## Cell 9 — STEP 05: Fill Missing Values

**Strategy per column:**
- `max_glu_serum` / `A1Cresult` → `'None'` — null means the test was simply not ordered, which is a valid clinical category
- `medical_specialty` → `'Unknown'` — specialty not documented at admission
- `payer_code` → `'Unknown'` — insurance not recorded
- `admission_source_id` ID=17, `admission_type_id` ID=6, `discharge_disposition_id` ID=18 → these are ICD-defined NULL codes, handled in the label mapping step
- `race` → mode (`Caucasian`) — small missingness (2.2%)
- `diag_1` → mode of primary diagnosis — only 10 rows missing
- `diag_2` / `diag_3` → `'0'` — no secondary/tertiary diagnosis recorded (clinically valid)

In [9]:
# max_glu_serum — 'None' means not tested
df['max_glu_serum'] = df['max_glu_serum'].fillna('None')
print(f"max_glu_serum distribution: {df['max_glu_serum'].value_counts().to_dict()}")

# A1Cresult — 'None' means not tested
df['A1Cresult'] = df['A1Cresult'].fillna('None')
print(f"A1Cresult distribution: {df['A1Cresult'].value_counts().to_dict()}")

# medical_specialty
df['medical_specialty'] = df['medical_specialty'].fillna('Unknown')
print(f"medical_specialty nulls remaining: {df['medical_specialty'].isnull().sum()}")

# payer_code
df['payer_code'] = df['payer_code'].fillna('Unknown')
print(f"payer_code nulls remaining: {df['payer_code'].isnull().sum()}")

# race — mode imputation
race_mode = df['race'].mode()[0]
df['race'] = df['race'].fillna(race_mode)
print(f"race: filled with mode '{race_mode}'. Nulls remaining: {df['race'].isnull().sum()}")

# diag_1 — primary diagnosis, 10 rows missing
diag1_mode = df['diag_1'].mode()[0]
df['diag_1'] = df['diag_1'].fillna(diag1_mode)
print(f"diag_1: filled with mode '{diag1_mode}'. Nulls remaining: {df['diag_1'].isnull().sum()}")

# diag_2 / diag_3 — '0' = no secondary/tertiary diagnosis
df['diag_2'] = df['diag_2'].fillna('0')
df['diag_3'] = df['diag_3'].fillna('0')
print(f"diag_2/diag_3 nulls remaining: {df['diag_2'].isnull().sum()} / {df['diag_3'].isnull().sum()}")

record(5, 'Filled all missing values: max_glu_serum/A1Cresult->None, specialty/payer->Unknown, race->mode, diag_2/3->0')
print(f'\nTotal nulls after step 05: {df.isnull().sum().sum()}')

max_glu_serum distribution: {'None': 66641, 'Norm': 1701, '>200': 936, '>300': 712}

A1Cresult distribution: {'None': 57144, '>8': 6239, 'Norm': 3741, '>7': 2866}
medical_specialty nulls remaining: 0
payer_code nulls remaining: 0
race: filled with mode 'Caucasian'. Nulls remaining: 0
diag_1: filled with mode '414'. Nulls remaining: 0
diag_2/diag_3 nulls remaining: 0 / 0
[STEP 05] Filled all missing values: max_glu_serum/A1Cresult->None, specialty/payer->Unknown, race->mode, diag_2/3->0



Total nulls after step 05: 0


## Cell 10 — STEP 06: Convert Age Bracket → Numeric Midpoint

**Why:** Age is stored as bracket strings like `[70-80)`. Converting to the numeric midpoint (75)
enables correlation analysis, regression modelling, and proper axis scaling in Tableau.
The original bracket string is preserved in a new `age_bracket` column.

In [10]:
AGE_MAP = {
    '[0-10)'  :  5,
    '[10-20)' : 15,
    '[20-30)' : 25,
    '[30-40)' : 35,
    '[40-50)' : 45,
    '[50-60)' : 55,
    '[60-70)' : 65,
    '[70-80)' : 75,
    '[80-90)' : 85,
    '[90-100)': 95
}

df['age_bracket'] = df['age']          # preserve original
df['age'] = df['age'].map(AGE_MAP)

unmapped = df['age'].isnull().sum()
if unmapped > 0:
    df['age'].fillna(df['age'].median(), inplace=True)
    print(f'  Warning: {unmapped} age values unmapped — filled with median')

print(f'age range: {df["age"].min()} – {df["age"].max()}')
print(f'age distribution:\n{df["age"].value_counts().sort_index().to_string()}')

record(6, 'age: bracket string -> numeric midpoint (e.g. [70-80) -> 75). Original kept in age_bracket')

age range: 5 – 95
age distribution:
age
5       153
15      534
25     1121
35     2692
45     6828
55    12351
65    15689
75    17751
85    11110
95     1761
[STEP 06] age: bracket string -> numeric midpoint (e.g. [70-80) -> 75). Original kept in age_bracket


## Cell 11 — STEP 07: Map IDs to Human-Readable Labels

**Why:** Convert integer codes for admission type, discharge disposition, and admission source into understandable categories based on the ICD-9 mapping. We replace the original columns to keep the feature space minimal.


In [11]:
ADMISSION_TYPE_MAP = {
    1: 'Emergency', 2: 'Urgent', 3: 'Elective', 4: 'Newborn', 
    5: 'Not Available', 6: 'NULL', 7: 'Trauma Center', 8: 'Not Mapped'
}

DISCHARGE_MAP = {
    1 : 'Discharged to Home', 2 : 'Discharged/Transferred to SNF',
    3 : 'Discharged/Transferred to SNF', 4 : 'Discharged/Transferred to ICF',
    5 : 'Discharged/Transferred to Another Facility', 6 : 'Home with Home Health',
    7 : 'Left AMA', 8 : 'Discharged/Transferred to Home IV', 9 : 'Admitted as Inpatient',
    10: 'Neonate/Transferred', 12: 'Still Patient', 15: 'Swing Bed', 16: 'Outpatient',
    17: 'Outpatient', 18: 'NULL', 22: 'Rehab Facility', 23: 'Long-term Care',
    24: 'Long-term Care', 25: 'Not Mapped', 26: 'Unknown', 27: 'Inpatient Rehab',
    28: 'Inpatient Rehab', 29: 'Outpatient Rehab', 30: 'Outpatient Rehab'
}

ADMISSION_SOURCE_MAP = {
    1 : 'Physician Referral', 2 : 'Clinic Referral', 3 : 'HMO Referral',
    4 : 'Transfer from Hospital', 5 : 'Transfer from SNF', 6 : 'Transfer from Facility',
    7 : 'Emergency Room', 8 : 'Court/Law Enforcement', 9 : 'Not Available',
    10: 'Transfer from Critical Access Hospital', 11: 'Normal Delivery',
    12: 'Premature Delivery', 13: 'Sick Baby', 14: 'Extramural Birth',
    15: 'Not Available', 17: 'NULL', 18: 'Transfer from Home Health',
    19: 'Readmission to Home Health', 20: 'Not Mapped', 21: 'Unknown',
    22: 'Transfer from Outpatient Surgery', 23: 'Born Inside this Hospital',
    24: 'Born Outside this Hospital', 25: 'Transfer from Ambulatory Surgery',
    26: 'Transfer from Hospice'
}

if 'admission_type_id' in df.columns:
    df['admission_type_id'] = df['admission_type_id'].map(ADMISSION_TYPE_MAP).fillna('Other')
if 'discharge_disposition_id' in df.columns:
    df['discharge_disposition_id'] = df['discharge_disposition_id'].map(DISCHARGE_MAP).fillna('Other')
if 'admission_source_id' in df.columns:
    df['admission_source_id'] = df['admission_source_id'].map(ADMISSION_SOURCE_MAP).fillna('Other')

record(7, 'Mapped admission/discharge IDs to human-readable strings in-place')



[STEP 07] Mapped admission/discharge IDs to human-readable strings in-place


## Cell 12 — STEP 08: Group ICD-9 Codes to Disease Categories

**Why:** Granular ICD-9 codes are too sparse for modeling/analysis. We group them into 10 broad clinical categories and replace the original columns.


In [12]:
def icd9_to_category(code):
    try:
        code = str(code).strip()
        if code.startswith('V') or code.startswith('E'): return 'External/Supplementary'
        num = float(code)
        if 250 <= num < 251: return 'Diabetes'
        elif 390 <= num < 460 or num == 785: return 'Circulatory'
        elif 460 <= num < 520 or num == 786: return 'Respiratory'
        elif 520 <= num < 580 or num == 787: return 'Digestive'
        elif 800 <= num < 1000: return 'Injury/Poisoning'
        elif 710 <= num < 740: return 'Musculoskeletal'
        elif 580 <= num < 630 or num == 788: return 'Genitourinary'
        elif 140 <= num < 240: return 'Neoplasms'
        elif 290 <= num < 320: return 'Mental Disorders'
        else: return 'Other'
    except:
        return 'Unknown'

for d_col in ['diag_1', 'diag_2', 'diag_3']:
    if d_col in df.columns:
        df[d_col] = df[d_col].apply(icd9_to_category)

record(8, 'Grouped diag_1/2/3 ICD-9 codes into broad disease categories in-place')



[STEP 08] Grouped diag_1/2/3 ICD-9 codes into broad disease categories in-place


## Cell 13 — STEP 09: Ordinal Encoding for Medications & Labs

**Why:** Convert string indicators ('No', 'Steady', 'Up', 'Down') to numeric ordinal values. This is essential for correlation analysis. Original columns are replaced in-place.


In [13]:
MED_MAP = {'No': 0, 'Steady': 1, 'Down': 2, 'Up': 3}
MED_COLS = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'glipizide', 'glyburide', 'tolbutamide',
    'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol',
    'troglitazone', 'tolazamide', 'insulin',
    'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone',
    'metformin-pioglitazone'
]

encoded_meds = 0
for col in MED_COLS:
    if col in df.columns:
        df[col] = df[col].map(MED_MAP).fillna(0).astype(int)
        encoded_meds += 1

GLU_SERUM_MAP = {'None': 0, 'Norm': 1, '>200': 2, '>300': 3}
A1C_MAP       = {'None': 0, 'Norm': 1, '>7'  : 2, '>8'  : 3}

if 'max_glu_serum' in df.columns:
    df['max_glu_serum'] = df['max_glu_serum'].map(GLU_SERUM_MAP).fillna(0).astype(int)
if 'A1Cresult' in df.columns:
    df['A1Cresult'] = df['A1Cresult'].map(A1C_MAP).fillna(0).astype(int)

record(9, f'Applied ordinal encoding to {encoded_meds} med columns and 2 lab columns in-place')



[STEP 09] Applied ordinal encoding to 20 med columns and 2 lab columns in-place


## Cell 14 — STEP 10: Create 30-Day Target and Standardize Columns

**Why:** Replace the multiclass `readmitted` column with a binary `readmitted_30day` target for binary classification, and standardize names.


In [14]:
if 'readmitted' in df.columns:
    df['readmitted_30day'] = (df['readmitted'] == '<30').astype(int)
    rate_30 = df['readmitted_30day'].mean() * 100
    df.drop(columns=['readmitted'], inplace=True)
    record(10, f'Replaced readmitted with binary readmitted_30day target. Rate = {rate_30:.2f}%')

# Standardize column names (snake_case)
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('-', '_', regex=False)
)

record(11, 'Standardized column names to lowercase snake_case')
print(f'Shape after step 10-11: {df.shape}')



[STEP 10] Replaced readmitted with binary readmitted_30day target. Rate = 8.98%
[STEP 11] Standardized column names to lowercase snake_case
Shape after step 10-11: (69990, 48)


## Cell 15 — Export Cleaned Dataset and Write Log

Final write step to `data/processed/`.


In [15]:
# Export cleaned CSV
df.to_csv(PROCESSED_PATH, index=False)
file_mb = os.path.getsize(PROCESSED_PATH) / 1e6

record(12, f'FINAL EXPORT: {PROCESSED_PATH} | {len(df):,} rows x {len(df.columns)} cols | {file_mb:.1f} MB')

print(f'Cleaned dataset saved to : {PROCESSED_PATH}')
print(f'File size                : {file_mb:.1f} MB')
print(f'Shape                    : {len(df):,} rows x {len(df.columns)} columns')

# Write transformation log
with open(LOG_PATH, 'w') as f:
    f.write('NST DVA Capstone 2 - ETL Transformation Log\n')
    f.write('Notebook : 02_cleaning.ipynb\n')
    f.write('Dataset  : Diabetic Patient Readmission (Healthcare)\n')
    f.write('=' * 60 + '\n\n')
    for entry in log:
        f.write(entry + '\n')
    f.write(f'\nTotal transformations: {len(log)}\n')

print(f'\nTransformation log saved : {LOG_PATH}')
print(f'Total steps logged       : {len(log)}')



[STEP 12] FINAL EXPORT: ../data/processed/diabetic_data_clean.csv | 69,990 rows x 48 cols | 15.2 MB
Cleaned dataset saved to : ../data/processed/diabetic_data_clean.csv
File size                : 15.2 MB
Shape                    : 69,990 rows x 48 columns

Transformation log saved : ../data/processed/02_cleaning_log.txt
Total steps logged       : 13


## Cell 16 — Cleaning Summary


In [16]:
print('=' * 80)
print('  ETL CLEANING SUMMARY')
print('=' * 80)

summary = [
    ['01', 'Dropped zero-variance columns', 'examide, citoglipton'],
    ['02', 'Dropped weight column', '96.9% missing - not reliable to impute'],
    ['03', 'Removed deceased/hospice rows', 'discharge_disposition_id in [11,13,14,19,20,21]'],
    ['04', 'Deduplicated to first encounter', 'one row per patient_nbr to reduce leakage'],
    ['05', 'Filled missing values', 'clinical/context-aware fills only'],
    ['06', 'Age bracket to numeric midpoint', 'in-place cleaning transformation'],
    ['07', 'Admission/Discharge IDs -> Labels', 'Mapped to human readable in-place'],
    ['08', 'Diag_1/2/3 -> Disease Categories', 'Grouped ICD-9 codes in-place'],
    ['09', 'Medications & Labs -> Ordinal', 'Encoded to integers in-place'],
    ['10', 'Readmitted -> Binary Target', 'Replaced with readmitted_30day'],
    ['11', 'Standardized column names', 'lowercase snake_case'],
    ['12', 'Exported cleaned dataset', 'saved into data/processed'],
]

print(f"{'Step':<5} {'Action':<35} {'Justification'}")
print('-' * 80)
for row in summary:
    print(f'{row[0]:<5} {row[1]:<35} {row[2]}')

print()
print(f'Input  : {RAW_ROWS:,} rows x {RAW_COLS} cols')
print(f'Output : {len(df):,} rows x {len(df.columns)} cols')
print(f'Nulls  : {df.isnull().sum().sum()}')
print('=' * 80)



  ETL CLEANING SUMMARY
Step  Action                              Justification
--------------------------------------------------------------------------------
01    Dropped zero-variance columns       examide, citoglipton
02    Dropped weight column               96.9% missing - not reliable to impute
03    Removed deceased/hospice rows       discharge_disposition_id in [11,13,14,19,20,21]
04    Deduplicated to first encounter     one row per patient_nbr to reduce leakage
05    Filled missing values               clinical/context-aware fills only
06    Age bracket to numeric midpoint     in-place cleaning transformation
07    Admission/Discharge IDs -> Labels   Mapped to human readable in-place
08    Diag_1/2/3 -> Disease Categories    Grouped ICD-9 codes in-place
09    Medications & Labs -> Ordinal       Encoded to integers in-place
10    Readmitted -> Binary Target         Replaced with readmitted_30day
11    Standardized column names           lowercase snake_case
12    Exported cl